# DPE Head Reweighting

This notebook provides a general Jupyter interface for the Waterbirds H2 experiment using precomputed raw ResNet-50 features. It delegates experiment logic to the repository's Python implementation.

**Prerequisites:** a writable repository checkout, Python 3.10 or later, and matching Stage-0 artifacts: `metadata.csv`, `ckpt_last.pt`, `val_raw.npz`, `test_raw.npz` and `provenance.json`. The checkpoint must use final-epoch selection without validation/test-based tuning. Backbone training and feature extraction are external prerequisites.

Feature NPZ files contain exactly `ids` (Unicode filenames in metadata split order) and `features` (raw floating-point arrays of shape `[1199, 2048]` for validation and `[5794, 2048]` for test). The provenance file identifies the track, seed, checkpoint and validation-feature hashes, transforms, pretrained weights and source commit. See the [input specification](../src/methods/dpe_head_reweighting/README.md#input-specification).

The validation split is divided into A for head fitting and epoch selection, B for reliability estimation, and C for aggregation selection. Test evaluation occurs only after selection is persisted. Standard and enhanced tracks are evaluated separately for seeds 0, 1 and 2. Synthetic CPU validation is available; real Waterbirds results remain pending.

In [ ]:
# Environment initialization. Set REPO_PATH if automatic discovery is insufficient.
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

REPO_PATH = None  # Optional explicit path to a writable repository checkout.

if REPO_PATH is not None:
    REPO = Path(REPO_PATH).expanduser().resolve()
else:
    working_dir = Path.cwd().resolve()
    candidates = [working_dir, *working_dir.parents]
    candidates += [working_dir / "waterbirds-shortcut-mitigation"]
    REPO = next((path for path in candidates
                 if (path / "scripts/dpe_head_reweighting.py").is_file()), None)
if REPO is None or not (REPO / "scripts/dpe_head_reweighting.py").is_file():
    raise FileNotFoundError("Set REPO_PATH to a writable waterbirds-shortcut-mitigation checkout.")

# Network access is required only when a dependency is missing.
missing = [package for module, package in
           [("numpy", "numpy"), ("torch", "torch"), ("yaml", "pyyaml"), ("matplotlib", "matplotlib")]
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repository:", REPO)
print("Python:", sys.version.split()[0])
# Existing checkouts are used as supplied; no revision changes are performed.
try:
    print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
except (OSError, subprocess.CalledProcessError):
    print("Git commit unavailable; implementation file hashes will still be recorded.")

## 1. Experiment configuration

`INPUT_DIR` identifies the matching artifacts for one track and Stage-0 seed. Individual filenames may be adjusted in `inputs`. `OUTPUT_DIR` must be writable. `RUN_NAME` identifies an independent experiment; a different configuration requires a distinct run directory.

The device is selected automatically from the available PyTorch backend. The YAML schema fixes the declared training recipe. Re-execution reuses completed stages only after validation of their associated inputs.

In [ ]:
import json
import torch
import yaml

TRACK = "standard"  # "standard" or "enhanced"
SEED = 0            # 0, 1 or 2; requires a matching Stage-0 seed
RUN_NAME = f"h2-{TRACK}-seed{SEED}"
INPUT_DIR = (REPO / "data/dpe_head_reweighting" / TRACK / f"seed{SEED}").resolve()
OUTPUT_DIR = (REPO / "outputs/dpe_head_reweighting_runs").resolve()

inputs = {
    "metadata": INPUT_DIR / "metadata.csv",
    "backbone": INPUT_DIR / "ckpt_last.pt",
    "val_features": INPUT_DIR / "val_raw.npz",
    "test_features": INPUT_DIR / "test_raw.npz",
    "provenance": INPUT_DIR / "provenance.json",
}
# Test features may be supplied later, before running the final test cell.
missing = [str(path) for key, path in inputs.items() if key != "test_features" and not path.is_file()]
if missing:
    raise FileNotFoundError("Required input artifacts are missing; configure INPUT_DIR:\n" + "\n".join(missing))

cfg = yaml.safe_load((REPO / f"configs/dpe_head_reweighting/waterbirds_{TRACK}.yaml").read_text())
cfg.update(track=TRACK, seed=SEED, device="cuda" if torch.cuda.is_available() else "cpu")
cfg.update({key: str(path) for key, path in inputs.items()})
RUN_DIR = OUTPUT_DIR / RUN_NAME
cfg.update(experiment_dir=str(RUN_DIR / "experiment"),
           ensemble_path=str(RUN_DIR / "checkpoints/ensemble.pt"),
           results_dir=str(RUN_DIR / "results"))
CONFIG_PATH = RUN_DIR / "config.yaml"
RUN_DIR.mkdir(parents=True, exist_ok=True)
resolved_yaml = yaml.safe_dump(cfg, sort_keys=False)
if CONFIG_PATH.exists() and CONFIG_PATH.read_text() != resolved_yaml:
    raise ValueError("This RUN_NAME already has a different configuration. Choose a new RUN_NAME.")
CONFIG_PATH.write_text(resolved_yaml)

# Imports refer to the selected repository checkout.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.methods.dpe_head_reweighting.experiment import config, context, digest, get_ensemble, lock, read_json
resolved = config(CONFIG_PATH)

def run(action):
    print(f"Running {action}: {TRACK}, seed {SEED}, {cfg['device']}", flush=True)
    subprocess.run([sys.executable, str(REPO / "scripts/dpe_head_reweighting.py"),
                    action, "--config", str(CONFIG_PATH)], cwd=REPO, check=True)

print("Outputs:", RUN_DIR)
print("Device:", cfg["device"])
print("Stage-0 provenance:")
print(inputs["provenance"].read_text())

## 2. Validation partition preparation

Preparation verifies artifact hashes and official split counts, then persists a group-stratified 60% A / 20% B / 20% C partition using split seed 0. Test features are not required at this stage.

In [ ]:
manifest_path = Path(resolved["experiment_dir"]) / "manifest.json"
if manifest_path.exists():
    context(resolved)  # Reject changed inputs instead of silently reusing them.
    print("Reusing verified manifest.")
else:
    run("prepare")
manifest = read_json(manifest_path)
print("Subset sizes:", {role: len(rows) for role, rows in manifest["parts"].items()})

## 3. Sequential head training

Fifteen heads are fitted using group-balanced samples from A, with 20 epochs per head. Epoch selection also uses A. Neither B, C nor test data is used for head fitting or epoch selection. The completed ensemble is exported for subsequent selection and evaluation.

In [ ]:
if Path(resolved["ensemble_path"]).exists():
    context(resolved)
    heads = get_ensemble(resolved)
    print("Reusing verified heads; selected epochs:", heads["selected_epochs"])
else:
    run("train-heads")

## 4. Reliability estimation and aggregation selection

Subset B determines per-head reliability scores and candidate weights. Subset C selects among uniform averaging and the 12 declared weighted configurations. The selection record contains all candidate metrics, selected weights and input identities. Uniform selection is retained as a valid outcome.

In [ ]:
selection_path = Path(resolved["experiment_dir"]) / "selection.json"
if selection_path.exists():
    context(resolved)
    selection = read_json(selection_path)
    if selection["lock"] != lock(resolved):
        raise ValueError("Saved selection inputs changed; use a fresh experiment.")
    print("Reusing frozen selection.")
else:
    run("fit-select")
selection = read_json(selection_path)
print(json.dumps(selection["selected"], indent=2))

## 5. Held-out evaluation

Evaluation requires a persisted selection record. The uniform control and selected aggregation are evaluated using identical test features and frozen heads. The predefined grid and partition remain fixed after test evaluation.

In [ ]:
metrics_path = Path(resolved["results_dir"]) / "metrics.json"
if metrics_path.exists():
    context(resolved)
    result = read_json(metrics_path)
    if result["lock"] != lock(resolved) or result["test_features_hash"] != digest(inputs["test_features"]) or result["metadata_hash"] != digest(inputs["metadata"]):
        raise ValueError("Existing test result inputs changed; preserve the original experiment.")
    print("Displaying existing final evaluation.")
else:
    if not inputs["test_features"].is_file():
        raise FileNotFoundError(f"Required test feature file is missing: {inputs['test_features']}")
    run("evaluate")
result = read_json(metrics_path)
for method in ("uniform", "selected"):
    m = result[method]
    print(f"{method:>8}: accuracy={m['overall']:.2%}, WGA={m['wga']:.2%}")
    print("          per-group accuracy:", m["groups"])
print("Paired changes (percentage points):", result["delta_pp"])
print("Effective head count:", result["effective_heads"])
from IPython.display import display, Image
plot = Path(resolved["results_dir"]) / "head_diagnostics.png"
display(Image(filename=str(plot)))

## 6. Artifact export

The archive contains the resolved configuration, partition manifest, trained heads, selection record, metrics and diagnostic plots. Available per-head prediction caches are included. Input checkpoints and raw features are excluded.

The ZIP is written beside the run directory under `OUTPUT_DIR`. The archive and execution-environment record should be retained with the experiment. Track-level summaries require the three seed results to be collected in the paths declared by the corresponding CLI configuration; standard and enhanced tracks are summarized separately.

In [ ]:
import platform
import shutil

# Record notebook execution metadata alongside the runner's artifact provenance.
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": str(torch.__version__),
    "device": cfg["device"],
    "accelerator": torch.cuda.get_device_name(0) if cfg["device"] == "cuda" else "CPU",
}
(RUN_DIR / "notebook_environment.json").write_text(json.dumps(environment, indent=2) + "\n")
cache_dir = REPO / "outputs/dpe_head_reweighting" / TRACK / f"seed{SEED}"
if cache_dir.exists():
    shutil.copytree(cache_dir, RUN_DIR / "predictions", dirs_exist_ok=True)
zip_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Archive:", zip_path)